# CRUD no Python - Integração Python e MySQL <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/mysql/mysql-original.svg" height="45" />🐬

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-mysql%20%7C%20crud-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc%20%7C%20sql-purple)
![Biblioteca](https://img.shields.io/badge/requer-mysql--connector--python-orange)

> Tudo que foi feito com SQL Server via `pyodbc` tem equivalente direto em **MySQL**, com uma biblioteca própria (`mysql-connector-python`) e uma sintaxe SQL quase idêntica. Este notebook refaz o CRUD inteiro — Create, Read, Update, Delete — agora contra um banco MySQL.

## 📋 Conteúdo

1. [pyodbc vs. mysql-connector-python](#-1-pyodbc-vs-mysql-connector-python)
2. [Conectando no MySQL](#-2-conectando-no-mysql)
3. [Create — Tabela e Registros](#-3-create-tabela-e-registros)
4. [Read](#-4-read)
5. [Update](#-5-update)
6. [Delete](#-6-delete)


## 🆚 1. pyodbc vs. mysql-connector-python

| Aspecto 🔑 | pyodbc + SQL Server 🔓 | mysql-connector-python 🔓 |
|---|---|---|
| Instalação | biblioteca Python + driver ODBC do sistema | só `pip install mysql-connector-python` (sem driver externo) |
| Conectar | `pyodbc.connect(string_de_conexao)` | `mysql.connector.connect(host=..., user=..., password=...)` |
| Parâmetro seguro | `?` | `%s` |
| Auto-incremento | `IDENTITY(1,1)` | `AUTO_INCREMENT` |
| Texto | `VARCHAR(n)` | `VARCHAR(n)` (igual) |

O fluxo (conectar → cursor → `execute()` → `commit()`/`fetchall()` → fechar) é exatamente o mesmo — só muda o dialeto.

## 🔌 2. Conectando no MySQL

Diferente do SQL Server, o MySQL cria o banco com `CREATE DATABASE` diretamente (sem precisar de um passo separado como o `master`).

In [1]:
import mysql.connector
from conexao import nova_conexao_mysql
from cores import *

conexao = nova_conexao_mysql()
cursor = conexao.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS HashtagCursoSQL")
conexao.database = "HashtagCursoSQL"

print(f"{VerdeClaro}Conectado ao MySQL, banco HashtagCursoSQL pronto.{Reset}")


Conectado ao MySQL, banco HashtagCursoSQL pronto.


## ➕ 3. Create — Tabela e Registros

Mesma lógica do SQL Server: dropar se existir, criar a tabela, inserir com `%s` no lugar do `?`.

In [2]:
cursor.execute("DROP TABLE IF EXISTS Clientes")
cursor.execute("""
CREATE TABLE Clientes (
    Id INT AUTO_INCREMENT PRIMARY KEY,
    Nome VARCHAR(100) NOT NULL,
    Email VARCHAR(100) NOT NULL,
    Cidade VARCHAR(50) NOT NULL,
    DataCadastro DATE NOT NULL
)
""")

clientes = [
    ("Marina Torres", "marina.torres@email.com", "Recife", "2026-01-10"),
    ("Felipe Nogueira", "felipe.nogueira@email.com", "São Paulo", "2026-01-12"),
    ("Juliana Prado", "juliana.prado@email.com", "Belo Horizonte", "2026-01-15"),
    ("Rafael Costa", "rafael.costa@email.com", "Recife", "2026-01-18"),
]

cursor.executemany(
    "INSERT INTO Clientes (Nome, Email, Cidade, DataCadastro) VALUES (%s, %s, %s, %s)",
    clientes
)
conexao.commit()

print(f"{VerdeClaro}Tabela Clientes criada com {len(clientes)} registros.{Reset}")


Tabela Clientes criada com 4 registros.


## 📖 4. Read

`fetchall()` funciona exatamente igual ao `pyodbc` — a diferença é que cada linha vem como tupla simples, não como objeto com atributo por nome (a não ser que o cursor seja criado com `dictionary=True`).

In [3]:
cursor.execute("SELECT Id, Nome, Cidade FROM Clientes WHERE Cidade = %s", ("Recife",))
clientes_recife = cursor.fetchall()

print(f"{CinzaClaro}Clientes de Recife:{Reset}")
for id_cliente, nome, cidade in clientes_recife:
    print(f"  #{id_cliente} {VerdeClaro}{nome}{Reset} — {cidade}")


Clientes de Recife:
  #1 Marina Torres — Recife
  #4 Rafael Costa — Recife


## ✏️ 5. Update

In [4]:
cursor.execute(
    "UPDATE Clientes SET Cidade = %s WHERE Nome = %s",
    ("Olinda", "Marina Torres")
)
conexao.commit()

cursor.execute("SELECT Nome, Cidade FROM Clientes WHERE Nome = %s", ("Marina Torres",))
nome, cidade_atualizada = cursor.fetchone()
print(f"{VerdeClaro}{nome}{Reset} agora está em {MagentaClaro}{cidade_atualizada}{Reset}")


Marina Torres agora está em Olinda


## 🗑️ 6. Delete

In [5]:
cursor.execute("DELETE FROM Clientes WHERE Nome = %s", ("Rafael Costa",))
conexao.commit()

cursor.execute("SELECT COUNT(*) FROM Clientes")
total_restante = cursor.fetchone()[0]

print(f"{VermelhoClaro}Cliente removido.{Reset} {CinzaClaro}Total restante:{Reset} {MagentaClaro}{total_restante}{Reset}")

cursor.close()
conexao.close()


Cliente removido. Total restante: 3


O CRUD em MySQL segue exatamente o mesmo raciocínio do SQL Server — a troca de driver (`pyodbc` → `mysql-connector-python`) e de placeholder (`?` → `%s`) é praticamente tudo que muda. É essa portabilidade de conceito, mais do que de código, que faz valer a pena aprender o padrão CRUD uma vez e reaplicar em qualquer banco relacional.

> ▶️ Próximo notebook: **Disponibilizando um Minicurso de SQL**.